# MHEMe Evaluation

## 0. Introduction

How well does our model perform? Let's discover it!

The goal of this notebook is to train several models for the several datasets, plots some results and try to understand if our model really stands out.

The models implemented will be characterized by:
1. General approach
    - MHEMe models
    - Direct models
2. Type of direct model
    - Temporary Convolutional Network (TCN)
    - XGBoost
    - ARIMA (only for baseline)
    - SARIMA (only for baseline)
3. [Type of training objective](Horizon%20Aware%20Loss.ipynb#Horizon-Aware-Huber-Loss)
    - Mean Squared Error (MSE)
    - Horizon Aware Huber Loss (HAH)
4. [Type of weighting strategy](Horizon%20Aware%20Loss.ipynb#Possible-weighting-strategies)
    - Uniform (U)
    - Soft Linear (SO)
    - Strong Linear (ST)
    - Exponential (E)

All the created models will be trained on a single time series from all the benchmark datasets presented in the [Exploratory Data Analysis notebook](eda.ipynb#Dataset-import-and-description), and will be tested using their respective loss as comparison metric.

See the quoted notebooks for more references.

The analysis will try to answer the following questions:
1. **Overall performance**: is using MHEMe actually *useful*?
2. **Variance of predictors**: are the single models learning something different?
3. **Relevance of weighting strategy**: does the choice affect the performance?
4. **Generalization**: are the dynamics learnt someway *general*?


## 1. Imports and Data Loading

In [1]:
# Imports
import pandas as pd
import numpy as np
import os

from src.data_handler import *
from src.model_handler import *
from src.config_files import *
from src.direct_models import *
from src.mheme import *

In [2]:
# Global variables
WINDOW = 48
HORIZON = 12

DATA_PATH = '../data'
DATA_CONFIG_PATH = '../data/data_config.json'

TCN_PATH_CONFIG_LOAD = '../src/config_files/tcn_config.json'
TCN_PATH_SAVE = '../models/'

XGB_PATH_CONFIG_LOAD = '../src/config_files/xgb_config.json'
XGB_PATH_SAVE = '../models/'

ARIMA_PATH_CONFIG_LOAD = '../src/config_files/arima_config.json'

MODELS_PATH_SAVE = '../models/'

## 2. Models training

### 2.0 Automatic Workflow

In [3]:
# PER RUNNARE AUTO_WORKFLOW
# ----------------------------------------
#
# 1. Scegli il dataset ('e', 's', 't', ...)
#
# 2. Setta a True le variabili relative ai modelli da trainare: 
# - XYZ_each indica il training di 100 versioni del modello XYZ, ciascuna su una time series diversa
# - XYZ_all indica il training di 1 versione del modello XYZ sull'insieme delle time series del modello
# - NB: arima_all non ha senso! è volutamente NON previsto
#
# 3. Definisci le proporzioni con prop(A, B, C), dimensioni di Train, Val, Test (devono sommare a 1).
# - ARIMA definisce solo train (dimensione Train+Val) e test(dimensione test)
#
# 4. shuffle_data: Consigliato FALSE. Shuffling delle finestre PRIMA di train/test split. Quindi il test NON è l'ultimo pezzo della time_series.
# 5. shuffle_internal : Consigliato TRUE. Shuffling delle finestre DOPO il train/test/split. Quindi il test è comunque relativo all'ultimo pezzo della time_series.
# 6. random_state: fissato, funziona sia per shuffle_data sia per shuffle_internal.
# 
# 7. skip_umheme: distanza di HORIZON per i diversi modelli di UMHEMe. Viene anche usato da ARIMA per capire quanti valori sovrascrivere in predizione
#
# 8. weight_type : usato SOLO da UMHEMe. LISTA! contenente (almeno) uno, ma anche più di uno, tra 'exp', 'soft_lin', 'strong_lin', 'uni'
# 9. loss_type : usato, in pratica, SOLO da UMHEMe. LISTA! contenente (almeno) uno, ma anche più di uno, tra 'horizon_weighted_huber', 'mse'
#
# Restituisce i MSE

In [ ]:
results = auto_workflow(dataset_init = 's',
                        data_path=DATA_PATH,
                        data_config_path=DATA_CONFIG_PATH,
                        model_config_paths=[TCN_PATH_CONFIG_LOAD, XGB_PATH_CONFIG_LOAD, ARIMA_PATH_CONFIG_LOAD],
                        tcn_each=False,
                        tcn_all=False,
                        xgb_each=False,
                        xgb_all=False,
                        umheme_each=False,
                        umheme_all=True,
                        arima=True,
                        prop= (0.7, 0.1, 0.2),
                        shuffle_data= False, 
                        shuffle_internal=False,
                        random_state= 42,
                        skip_umheme = 5,
                        weight_type = ['exp'],
                        loss_type = ['horizon_weighted_huber'],
                        pretrained_model=False
                    )


--------------------
Processing time series with ID:	0
--------------------
Configuration file ../data/data_config.json modified


C:\Users\Utente\Documents\Università\Multiple Horizons Ensemble Method\src\data_handler.py:190: FutureWarning: 'T' is deprecated and will be removed in a future version, please use 'min' instead.
  df.index = pd.date_range(start="2000-01-01", periods=len(df), freq=base_freq)


Configuration file ../src/config_files/arima_config.json modified
Configuration file ../src/config_files/arima_config.json modified


Train model: ARIMA_mse_uni
ARIMA model parameters saved to ../models/solar/arima/0/ARIMA_mse_uni_s.pkl


Test set evaluation:

Evaluate model: ARIMA_mse_uni

--------------------
Processing time series with ID:	1
--------------------
Configuration file ../data/data_config.json modified


C:\Users\Utente\Documents\Università\Multiple Horizons Ensemble Method\src\data_handler.py:190: FutureWarning: 'T' is deprecated and will be removed in a future version, please use 'min' instead.
  df.index = pd.date_range(start="2000-01-01", periods=len(df), freq=base_freq)


Configuration file ../src/config_files/arima_config.json modified
Configuration file ../src/config_files/arima_config.json modified


Train model: ARIMA_mse_uni
ARIMA model parameters saved to ../models/solar/arima/1/ARIMA_mse_uni_s.pkl


Test set evaluation:

Evaluate model: ARIMA_mse_uni

--------------------
Processing time series with ID:	2
--------------------
Configuration file ../data/data_config.json modified


...

Configuration file ../src/config_files/arima_config.json modified
Configuration file ../src/config_files/arima_config.json modified


Train model: ARIMA_mse_uni
ARIMA model parameters saved to ../models/solar/arima/97/ARIMA_mse_uni_s.pkl


Test set evaluation:

Evaluate model: ARIMA_mse_uni

--------------------
Processing time series with ID:	98
--------------------
Configuration file ../data/data_config.json modified


C:\Users\Utente\Documents\Università\Multiple Horizons Ensemble Method\src\data_handler.py:190: FutureWarning: 'T' is deprecated and will be removed in a future version, please use 'min' instead.
  df.index = pd.date_range(start="2000-01-01", periods=len(df), freq=base_freq)


Configuration file ../src/config_files/arima_config.json modified
Configuration file ../src/config_files/arima_config.json modified


Train model: ARIMA_mse_uni
ARIMA model parameters saved to ../models/solar/arima/98/ARIMA_mse_uni_s.pkl


Test set evaluation:

Evaluate model: ARIMA_mse_uni

--------------------
Processing time series with ID:	99
--------------------
Configuration file ../data/data_config.json modified


C:\Users\Utente\Documents\Università\Multiple Horizons Ensemble Method\src\data_handler.py:190: FutureWarning: 'T' is deprecated and will be removed in a future version, please use 'min' instead.
  df.index = pd.date_range(start="2000-01-01", periods=len(df), freq=base_freq)


Configuration file ../src/config_files/arima_config.json modified
Configuration file ../src/config_files/arima_config.json modified


Train model: ARIMA_mse_uni
ARIMA model parameters saved to ../models/solar/arima/99/ARIMA_mse_uni_s.pkl


Test set evaluation:

Evaluate model: ARIMA_mse_uni
Results (CSV) saved to: ../models/results/errors_s_20260201_213633.csv
Results (JSON) saved to: ../models/results/errors_s_20260201_213633.json


In [5]:
# TRAIN STRUCTURE IF NOT USING AUTO_WORKFLOW
# 
# Define models
# models = models_definer(dataset_init = 'e', loss_type = 'horizon_weighted_huber', model_type = ['TCN', 'XGBoost'], weights_type = 'uni', config_model_paths = [TCN_PATH_CONFIG_LOAD, XGB_PATH_CONFIG_LOAD])
#          LOSS TYPES: 'mse', 'horizon_weighted_huber'
#          MODEL TYPES: 'TCN', 'XGBoost', 'ARIMA', 'UMHEMe'
#          WEIGHTS TYPES: 'uni', 'soft_lin', 'strong_lin', 'exp'
# 
# Train models
# models_trainer(models = models, train = train_DATASET-INITIAL-LETTER, dataset_init = DATASET-INITIAL-LETTER, save_models = True, path_save = MODELS_PATH_SAVE)
# 
# Fit models
# models_evaluator(models = models, test = test_DATASET-INITIAL-LETTER, dataset_init = DATASET-INITIAL-LETTER)

In [6]:
# IF NOT USING AUTO_WORKFLOW

dataset_initials = {'e': 'electricity', 's': 'solar', 't': 'traffic', 'v': 'volatility', 'w': 'wind'}
# dataset_initials = {'e': 'electricity'}

for ds in dataset_initials:
    print(f"Loading dataset: {dataset_initials[ds]}")

    train, val, test, X, data = dataset_handler(dataset_init = ds,
                                                data_path = DATA_PATH,
                                                data_config_path = DATA_CONFIG_PATH,
                                                is_arima = True,
                                                prop = [0.9, 0.08, 0.02],
                                                shuffle_data = False,
                                                shuffle_internal=True,
                                                random_state = 42
                                                ) 
    # If there exist a model with id_target == 'ALL' in data_config.json, then is much better to shuffle data, always with a fixed random state!
    
    # Change name of X, data based on ds. useful so that one can call 'train_e', 'val_e', 'test_e', .... with each dataset_init
    globals()[f'train_{ds}'] = train
    globals()[f'val_{ds}'] = val
    globals()[f'test_{ds}'] = test

    globals()[f'X_{ds}'] = X
    globals()[f'data_{ds}'] = data


Loading dataset: electricity
Loading dataset: solar
(52560, 100)
Loading dataset: traffic
Loading dataset: volatility


C:\Users\Utente\Documents\Università\Multiple Horizons Ensemble Method\src\data_handler.py:190: FutureWarning: 'T' is deprecated and will be removed in a future version, please use 'min' instead.
  df.index = pd.date_range(start="2000-01-01", periods=len(df), freq=base_freq)


Loading dataset: wind


### 2.1 Electricity Dataset

In [7]:
# Define models to evaluate
models = models_definer(dataset_init = ['e'],
                        loss_type = ['horizon_weighted_huber'],
                        model_type = ['ARIMA'],
                        weight_type = ['uni'],
                        data_config_path = DATA_CONFIG_PATH,
                        model_config_paths = [ARIMA_PATH_CONFIG_LOAD]
                        )

# Train models
models_trainer(models = models, train = train_e, dataset_init = 'e', save_models = True, models_path_save = MODELS_PATH_SAVE)

# Test models
eval_results_e = models_evaluator(models = models, test = test_e, dataset_init = 'e')

Configuration file ../src/config_files/arima_config.json modified


Train model: ARIMA_horizon_weighted_huber_uni


c:\Users\Utente\Documents\Università\Multiple Horizons Ensemble Method\.venv\Lib\site-packages\statsmodels\tsa\statespace\sarimax.py:978: UserWarning: Non-invertible starting MA parameters found. Using zeros as starting parameters.
  warn('Non-invertible starting MA parameters found.'


ARIMA model parameters saved to ../models//ARIMA_horizon_weighted_huber_uni_e.pkl


Test set evaluation:

Evaluate model: ARIMA_horizon_weighted_huber_uni


c:\Users\Utente\Documents\Università\Multiple Horizons Ensemble Method\.venv\Lib\site-packages\numpy\_core\fromnumeric.py:3824: RuntimeWarning: Mean of empty slice
  return _methods._mean(a, axis=axis, dtype=dtype,
c:\Users\Utente\Documents\Università\Multiple Horizons Ensemble Method\.venv\Lib\site-packages\numpy\_core\_methods.py:142: RuntimeWarning: invalid value encountered in scalar divide
  ret = ret.dtype.type(ret / rcount)


### 2.2 Solar Dataset

In [8]:
# Define models to evaluate
models = models_definer(dataset_init = ['s'],
                        loss_type = ['horizon_weighted_huber'],
                        model_type = ['TCN'],
                        weight_type = ['uni'],
                        data_config_path = DATA_CONFIG_PATH,
                        model_config_paths = [TCN_PATH_CONFIG_LOAD]
                        )

# Train models
models_trainer(models = models, train = train_s, dataset_init = 's', save_models = True, models_path_save = MODELS_PATH_SAVE)

# Test models
eval_results_s = models_evaluator(models = models, test = test_s, dataset_init = 's')

Configuration file ../src/config_files/tcn_config.json modified


Train model: TCN_horizon_weighted_huber_uni


TypeError: must be real number, not NoneType

### 2.3 Traffic Dataset

In [ ]:
# Define models to evaluate
models = models_definer(dataset_init = ['t'], loss_type = ['horizon_weighted_huber'], model_type = ['TCN'], weight_type = ['uni'], data_config_path = DATA_CONFIG_PATH, model_config_paths = [TCN_PATH_CONFIG_LOAD])

# Train models
models_trainer(models = models, train = train_t, dataset_init = 't', save_models = True, models_path_save = MODELS_PATH_SAVE)

# Test models
eval_results_t = models_evaluator(models = models, test = test_t, dataset_init = 't')

Configuration file ../src/config_files/tcn_config.json modified


Train model: TCN_horizon_weighted_huber_uni


Training TCN: 100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████| 20/20 [00:06<00:00,  2.89it/s]



Test set evaluation:

Evaluate model: TCN_horizon_weighted_huber_uni


### 2.4 Volatility Dataset

In [ ]:
# Define models to evaluate
models = models_definer(dataset_init = ['v'], loss_type = ['mse'], model_type = ['XGBoost'], weight_type = ['uni'], data_config_path = DATA_CONFIG_PATH, model_config_paths = [XGB_PATH_CONFIG_LOAD])

# Train models
models_trainer(models = models, train = train_v, dataset_init = 'v', save_models = True, models_path_save = MODELS_PATH_SAVE)

# Test models
eval_results_v = models_evaluator(models = models, test = test_v, dataset_init = 'v')

Configuration file ../src/config_files/xgb_config.json modified


Train model: XGBoost_mse_uni


Test set evaluation:

Evaluate model: XGBoost_mse_uni


/u/ibuttignon/.local/lib/python3.12/site-packages/xgboost/sklearn.py:1118: UserWarning: [17:52:58] WARNING: /workspace/src/c_api/c_api.cc:1575: Saving model in the UBJSON format as default.  You can use a file extension: `json` or `ubj` to choose between formats.
  self.get_booster().save_model(fname)


### 2.5 Wind Dataset

In [ ]:
# Define models to evaluate
models = models_definer(dataset_init = ['w'], loss_type = ['horizon_weighted_huber'], model_type = ['UMHEMe'], weight_type = ['uni', 'soft_lin', 'strong_lin', 'exp'], data_config_path = DATA_CONFIG_PATH, model_config_paths = [TCN_PATH_CONFIG_LOAD, XGB_PATH_CONFIG_LOAD], base_model = 'TCN')

# Train models
models_trainer(models = models, train = train_w, dataset_init = 'w', save_models = True, models_path_save = MODELS_PATH_SAVE)

# Test models
eval_results_w = models_evaluator(models = models, test = test_w, dataset_init = 'w')

Configuration file ../src/config_files/tcn_config.json modified
Configuration file ../src/config_files/tcn_config.json modified
Configuration file ../src/config_files/tcn_config.json modified
Configuration file ../src/config_files/tcn_config.json modified


Train model: UMHEMe_horizon_weighted_huber_uni
Fitting class : <class 'src.direct_models.TCN'>; horizon : 1


Training TCN: 100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████| 20/20 [00:16<00:00,  1.24it/s]


Fitting class : <class 'src.direct_models.TCN'>; horizon : 2


Training TCN: 100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████| 20/20 [00:16<00:00,  1.24it/s]


Fitting class : <class 'src.direct_models.TCN'>; horizon : 3


...

Fitting class : <class 'src.direct_models.TCN'>; horizon : 13


Training TCN: 100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████| 20/20 [00:16<00:00,  1.23it/s]


Fitting class : <class 'src.direct_models.TCN'>; horizon : 14


Training TCN: 100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████| 20/20 [00:16<00:00,  1.23it/s]




Test set evaluation:

Evaluate model: UMHEMe_horizon_weighted_huber_uni

Evaluate model: UMHEMe_horizon_weighted_huber_soft_lin

Evaluate model: UMHEMe_horizon_weighted_huber_strong_lin

Evaluate model: UMHEMe_horizon_weighted_huber_exp


## 3. Evaluations & Plots

### 3.0 Load models

In [ ]:
# For each model in the given folder, load the model.
model_folders = {'e': '../models/electricity/', 's': '../models/solar/', 't': '../models/traffic/', 'v': '../models/volatility/', 'w': '../models/wind/'}

for key, model_folder in model_folders.items():
    loaded_models = {}
    for model_file in os.listdir(model_folder):
        if model_file.endswith('.pkl'):
            model_name = model_file[:-4]  # Remove .pkl extension
            model_path = os.path.join(model_folder, model_file)
            loaded_model = UMHEMe.load_model(model_path)
            loaded_models[model_name] = loaded_model

    # Store loaded models in a global dictionary for later use
    globals()[f'models_{key}'] = loaded_models

### 3.1 Overall Performance

In [ ]:
# Evaluate the models on the test set

# Electricity
for model in models_e:
    print(f"\n\nEvaluating model: {model}")
    test = globals()[f'test_e']
    predictions = models_e[model].predict(test[0])
    mse = mean_squared_error(test[1], predictions)
    print(f"MSE for model {model}: {mse}")

# Solar
for model in models_s:
    print(f"\n\nEvaluating model: {model}")
    test = globals()[f'test_s']
    predictions = models_s[model].predict(test[0])
    mse = mean_squared_error(test[1], predictions)
    print(f"MSE for model {model}: {mse}")

# Traffic
for model in models_t:
    print(f"\n\nEvaluating model: {model}")
    test = globals()[f'test_t']
    predictions = models_t[model].predict(test[0])
    mse = mean_squared_error(test[1], predictions)
    print(f"MSE for model {model}: {mse}")

# Volatility
for model in models_v:
    print(f"\n\nEvaluating model: {model}")
    test = globals()[f'test_v']
    predictions = models_v[model].predict(test[0])
    mse = mean_squared_error(test[1], predictions)
    print(f"MSE for model {model}: {mse}")

# Wind
for model in models_w:
    print(f"\n\nEvaluating model: {model}")
    test = globals()[f'test_w']
    predictions = models_w[model].predict(test[0])
    mse = mean_squared_error(test[1], predictions)
    print(f"MSE for model {model}: {mse}")

### 3.2 Variance of Predictors 

### 3.3 Weighting Strategy Relevance

### 3.4 Generalization